# 09.2 - Tokens & Tokenization

**Phase:** 09 - Generative AI

**Status:** VERIFIED

---

## 1. What Are We Solving?

Tokenization converts raw text into discrete units (tokens) a model can process. Tokens are usually **subwords** - pieces of words - not whole words or characters. Tokenization determines what the model sees, how much you pay (API billing is per token), and how well the model handles non-English text, code, and special characters.

## 2. Why Does This Matter?

Misunderstanding tokenization causes budget surprises (1 word != 1 token), context-window overflows, and poor performance on non-English or code text. Tokenization is the bridge between human text and model-processable numbers.

## 3. Prerequisites

- Unit 09.1 (language model basics)

## 4. Learning Objectives

- Distinguish tokens, words, and characters
- Explain Byte Pair Encoding (BPE) at a high level
- Build a toy tokenizer from scratch
- Estimate token count and API cost

## 5. Mental Model

Tokenization is like taking apart a puzzle. The tokenizer breaks text into the smallest meaningful pieces it was trained on: whole words (`hello`), subwords (`un` + `likely`), or single characters. Common words become single tokens; rare words split into several pieces.

```text
"Tokenization is important"
      |
      v
['Token', 'ization', ' is', ' important']   <- tokens
    IDs:  [481,    ...   , ...   ]           <- numeric IDs the model sees
```


## 6. Setup

We implement tokenization by hand so no model weights or tokenizer files need downloading.


In [1]:
import matplotlib
matplotlib.use('Agg')
import re
from collections import Counter, defaultdict


## 7. Words vs Characters vs Tokens

The same text has a different unit count depending on the granularity you choose.


In [2]:
text = "Tokenization is surprisingly important for LLMs."

words = text.split()
chars = list(text)
print(f"Characters: {len(chars)}")
print(f"Words:      {len(words)}")
print(f"Words list: {words}")
print("\nAPI bills per *token* (a rough guide: ~4 chars or ~0.75 words per token).")


Characters: 48
Words:      6
Words list: ['Tokenization', 'is', 'surprisingly', 'important', 'for', 'LLMs.']

API bills per *token* (a rough guide: ~4 chars or ~0.75 words per token).


## 8. A Character-Level Tokenizer

Simplest tokenizer: map each character to an integer ID. Easy to build, but word lengths become very long sequences.


In [3]:
class CharTokenizer:
    def __init__(self, corpus):
        chars = sorted(set(corpus))
        self.stoi = {c: i for i, c in enumerate(chars)}
        self.itos = {i: c for c, i in self.stoi.items()}

    def encode(self, text):
        return [self.stoi[c] for c in text]

    def decode(self, ids):
        return ''.join(self.itos[i] for i in ids)

    @property
    def vocab_size(self):
        return len(self.stoi)

corpus = "the quick brown fox jumps over the lazy dog"
ct = CharTokenizer(corpus)
print("Vocab size:", ct.vocab_size)  # unique characters (+ space)
ids = ct.encode("fox jumps")
print("Encoded IDs:", ids)
print("Decoded:", repr(ct.decode(ids)))
print("7 characters -> 7 tokens (very inefficient for long text)")


Vocab size: 27
Encoded IDs: [6, 15, 24, 0, 10, 21, 13, 16, 19]
Decoded: 'fox jumps'
7 characters -> 7 tokens (very inefficient for long text)


## 9. Byte Pair Encoding (BPE) - The Real Approach

BPE steps:
1. Start with a vocabulary of individual characters.
2. Repeatedly count adjacent token pairs in the corpus.
3. Merge the most frequent pair into one new token.
4. Stop when the vocabulary reaches the desired size.

Common words become single tokens; rare words stay split. We implement a toy BPE.


In [4]:
def get_stats(tokens):
    pairs = defaultdict(int)
    for word in tokens:
        for i in range(len(word) - 1):
            pairs[(word[i], word[i+1])] += 1
    return pairs

def merge(tokens, pair, new_token):
    out = []
    for word in tokens:
        w = []
        i = 0
        while i < len(word):
            if i < len(word) - 1 and (word[i], word[i+1]) == pair:
                w.append(new_token)
                i += 2
            else:
                w.append(word[i])
                i += 1
        out.append(w)
    return out

words = [['t','h','e'],[],['t','h','e']]
words = [list(w) for w in "the the cat in the hat".split()]
print("Initial word splits:", words)

# Run 5 BPE merges
for step in range(5):
    stats = get_stats(words)
    if not stats:
        break
    best = max(stats, key=stats.get)
    new_tok = best[0] + best[1]
    words = merge(words, best, new_tok)
    print(f"step {step}: merged {best} -> '{new_tok}' (count {stats[best]})")

print("\nFinal tokenized words:", words)


Initial word splits: [['t', 'h', 'e'], ['t', 'h', 'e'], ['c', 'a', 't'], ['i', 'n'], ['t', 'h', 'e'], ['h', 'a', 't']]
step 0: merged ('t', 'h') -> 'th' (count 3)
step 1: merged ('th', 'e') -> 'the' (count 3)
step 2: merged ('a', 't') -> 'at' (count 2)
step 3: merged ('c', 'at') -> 'cat' (count 1)
step 4: merged ('i', 'n') -> 'in' (count 1)

Final tokenized words: [['the'], ['the'], ['cat'], ['in'], ['the'], ['h', 'at']]


## 10. Cost Estimation

In production, count tokens with the *exact* tokenizer for your model and multiply by the price per token.


In [5]:
# Toy estimate: English ~0.75 tokens per word on average
def estimate_tokens(text):
    return int(len(text.split()) / 0.75)

sample = "This paragraph is a typical user request sent to a language model API." * 5
est = estimate_tokens(sample)
price_per_1k = 0.002   # $ per 1K tokens (example)
cost = est / 1000 * price_per_1k
print(f"Estimated tokens: {est}")
print(f"Estimated cost:   ${cost:.5f}")
print("\nTip: always use the real tokenizer (tiktoken / model tokenizer) for exact budgets.")


Estimated tokens: 81
Estimated cost:   $0.00016

Tip: always use the real tokenizer (tiktoken / model tokenizer) for exact budgets.


## 11. Whitespace & Special Tokens

Real tokenizers are whitespace-sensitive: 'hello world' and 'helloworld' tokenize differently. Models also use special tokens (BOS, EOS, PAD, UNK, SEP, CLS) so the model knows stream boundaries.


In [6]:
# Demonstrating whitespace sensitivity in our char tokenizer
print("'hello world' ids:", ct.encode("hello world"))
print("'helloworld' ids:", ct.encode("helloworld"))
print("Same letters, different token sequences -> the model sees different input.")

# Special tokens are extra reserved IDs
SPECIALS = {"<BOS>": 0, "<EOS>": 1, "<PAD>": 2, "<UNK>": 3}
print("\nSpecial tokens reserve IDs:", SPECIALS)


'hello world' ids: [8, 5, 12, 12, 15, 0, 23, 15, 18, 12, 4]
'helloworld' ids: [8, 5, 12, 12, 15, 23, 15, 18, 12, 4]
Same letters, different token sequences -> the model sees different input.

Special tokens reserve IDs: {'<BOS>': 0, '<EOS>': 1, '<PAD>': 2, '<UNK>': 3}


## 12. Failure Case & Debugging

| Symptom | Cause | Fix |
|---|---|---|
| Cost higher than expected | More tokens than estimated | Count with real tokenizer |
| Non-English text performs poorly | Tokenizer makes many small tokens | Use multilingual tokenizer/model |
| Context window exceeded | Under-estimated tokens | Summarize / truncate |
| Special characters break | Unexpected tokenization | Inspect tokenized output |

## 13. Common Mistakes

- Assuming 1 word = 1 token.
- Mixing tokenizers from different models (IDs are meaningless across models).
- Ignoring whitespace sensitivity.
- Not counting tokens before calling paid APIs.

## 14. When NOT to Use a Custom Tokenizer

- Use the model's own pretrained tokenizer - the model was trained on those exact IDs.
- Build a custom one only for teaching or for unusual low-resource data.

## 15. Challenge

Extend the toy BPE to stop at a fixed vocab size and return the final token IDs for a held-out sentence.


In [7]:
def bpe_fit(corpus, num_merges=8):
    tokens = [list(w) for w in corpus.split()]
    merges = []
    for _ in range(num_merges):
        stats = get_stats(tokens)
        if not stats:
            break
        best = max(stats, key=stats.get)
        merges.append(best)
        tokens = merge(tokens, best, best[0] + best[1])
    return tokens, merges

def bpe_encode_corpus(text, merges):
    tokens = [list(w) for w in text.split()]
    for pair in merges:
        tokens = merge(tokens, pair, pair[0] + pair[1])
    return tokens

train = "low lower lowest newer newest"
_, merges = bpe_fit(train, num_merges=6)
print("Learned merges:", merges)
print("Encode 'lowest':", bpe_encode_corpus("lowest", merges))
print("-> 'low', 'est' became separate tokens, just like a real BPE model.")


Learned merges: [('w', 'e'), ('l', 'o'), ('lo', 'we'), ('s', 't'), ('n', 'e'), ('ne', 'we')]
Encode 'lowest': [['lowe', 'st']]
-> 'low', 'est' became separate tokens, just like a real BPE model.


## 16. Closed-Book Recall

1. What is the difference between a token and a word?
2. Why do different models use different tokenizers?
3. How does tokenization affect API cost?
4. What are special tokens and why do models need them?

## 17. Teach-Back Questions

Explain to another person:

- The BPE algorithm in 3-4 plain sentences.
- Why whitespace sensitivity matters.

## 18. Summary

You built a character tokenizer and a toy BPE tokenizer, learned about whitespace sensitivity and special tokens, and estimated API cost from token counts.

## 19. Further Experiment

- Compare token counts at character vs word vs subword granularity on the same text.
- Research how tiktoken counts tokens for a known sentence.

## 20. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: none (pure python)
OUTPUTS: PASS
LAST VERIFIED: 2026-08-29
```
